# 第2章 神经网络基础

本章将通过动手实践来理解神经网络的核心概念：

1. **从零实现单层感知器** —— 使用 NumPy 手写一个最简单的神经网络，理解前向传播、误差计算和梯度下降
2. **常见激活函数** —— 学习 Sigmoid、Tanh、ReLU 三种激活函数的原理与特点
3. **激活函数可视化对比** —— 直观感受不同激活函数的差异

> 每个代码单元格都可以独立运行，只需确保已安装对应的库。

## 背景：从 MP 模型到感知机

### MP 模型（McCulloch-Pitts 模型，1943）

MP 模型是由神经科学家 **Warren McCulloch** 和数学家 **Walter Pitts** 于 1943 年提出的第一个数学化的神经元模型，是所有人工神经网络的鼻祖。

**核心思想：** 用一个简单的数学模型来模拟生物神经元的工作方式——接收多个输入信号，经过加权求和后，通过一个阈值函数决定是否"激活"（输出 1 或 0）。

$$
y = f\left(\sum_{i=1}^{n} w_i x_i - \theta\right)
$$

其中：
- $x_i$ 为输入信号（0 或 1）
- $w_i$ 为对应的权重（在原始 MP 模型中权重是**固定的**，不可学习）
- $\theta$ 为阈值（threshold）
- $f(\cdot)$ 为阶跃函数：当输入 $\geq 0$ 时输出 1，否则输出 0

**MP 模型的特点：**

| 特点 | 说明 |
|------|------|
| 输入为二值 | 只接受 0 或 1 |
| 权重固定 | 需要人工设定，**不能自动学习** |
| 阶跃激活 | 输出也是 0 或 1，非黑即白 |
| 能力有限 | 可以实现 AND、OR 等简单逻辑门，但无法实现 XOR |

尽管 MP 模型本身不能学习，但它首次证明了**神经元的计算可以用数学来描述**，为后来的神经网络研究奠定了基础。

---

### 感知机（Perceptron，1957）

感知机由 **Frank Rosenblatt** 于 1957 年提出，是在 MP 模型基础上最重要的改进——**引入了自动学习机制**。

**与 MP 模型的核心区别：** 感知机的权重不再是手动设定的，而是通过训练数据**自动调整**。

$$
\hat{y} = f\left(\sum_{i=1}^{n} w_i x_i + b\right)
$$

其中 $b$ 为偏置（bias），$f$ 仍为阶跃函数。

**感知机学习规则：**

$$
w_i \leftarrow w_i + \eta \cdot (y - \hat{y}) \cdot x_i
$$

- $\eta$ 为学习率（learning rate），控制每次更新的步幅
- $y$ 为真实标签，$\hat{y}$ 为预测值
- 当预测正确时 $(y - \hat{y}) = 0$，权重不更新；预测错误时自动调整

**感知机 vs MP 模型：**

| | MP 模型 (1943) | 感知机 (1957) |
|------|------|------|
| 权重 | 手动设定 | 自动学习 |
| 学习能力 | 无 | 有（感知机学习规则） |
| 输入 | 仅二值 | 可以是实数 |
| 意义 | 数学建模的起点 | 机器学习的起点 |

**感知机的局限：** 1969 年，Minsky 和 Papert 在《Perceptrons》一书中证明了**单层感知机无法解决 XOR（异或）问题**——因为 XOR 不是线性可分的。这一结论导致了神经网络研究进入了长达十余年的"寒冬"。

**突破之路：** 解决 XOR 问题需要**多层感知机（MLP）**配合**反向传播算法**，这正是后来深度学习的基础。本章接下来的代码实践，就是从最简单的单层感知机开始，逐步理解这些核心概念。

## 1.1 从零实现单层神经网络

下面用 **纯 NumPy** 实现一个最简单的单层感知器（Single-Layer Perceptron），演示神经网络学习的完整过程。

### 核心步骤

| 步骤 | 说明 |
|------|------|
| ① 初始化 | 随机生成权重 $\mathbf{w}$（3×1 向量） |
| ② 前向传播 | 计算 $\hat{y} = \sigma(\mathbf{X} \cdot \mathbf{w})$，其中 $\sigma$ 为 Sigmoid 函数 |
| ③ 计算误差 | $e = y - \hat{y}$ |
| ④ 反向传播 | 梯度 $\delta = e \odot \sigma'(\hat{y})$，其中 $\sigma'(x) = x(1-x)$ |
| ⑤ 更新权重 | $\mathbf{w} \leftarrow \mathbf{w} + \mathbf{X}^T \cdot \delta$ |
| ⑥ 重复 | 迭代 10000 次 |

### 数据说明

- 输入 $\mathbf{X}$：4 个样本，每个样本 3 个特征
- 标签 $y$：对应的期望输出（0 或 1）

In [1]:
# -*- coding: utf-8 -*-

import numpy as np
from numpy import array, exp, random, dot

# 4个样本，3个特征的输入数据（XOR类似，但不是标准XOR）
x = array([
    [0, 0, 1],
    [1, 1, 1],
    [1, 0, 1],
    [0, 1, 1]
])

# 对应的期望输出（标签），列向量
y = array([
    [0],
    [1],
    [1],
    [0]
])  # .T 在原代码里写在这里，但其实可以直接写成列向量

# 设置随机种子，保证每次运行结果相同（方便教学/调试）
random.seed(1)

# 初始化权重：3×1 的列向量，范围 [-1, 1)
weights = 2 * random.random((3, 1)) - 1

# 梯度下降训练 10000 次
for it in range(10000):
    # 前向传播
    output = 1 / (1 + exp(-dot(x, weights)))          # sigmoid(x @ w)

    # 计算误差（预测值 - 真实值）
    error = y - output

    # 计算梯度（sigmoid的导数 * 误差）
    delta = error * output * (1 - output)             # 误差项 × sigmoid导数

    # 权重更新（梯度上升，注意正负号）
    weights += dot(x.T, delta)                        # Δw = X^T · δ

# 训练结束后打印最终权重
print("训练后的权重（3×1）：")
print(weights)

# 如果想看最终预测结果，可以再加这几行：
print("\n最终预测输出（四舍五入后）：")
print(np.round(output, 3))
print("\n真实标签：")
print(y.T)

训练后的权重（3×1）：
[[ 9.67299303]
 [-0.2078435 ]
 [-4.62963669]]

最终预测输出（四舍五入后）：
[[0.01 ]
 [0.992]
 [0.994]
 [0.008]]

真实标签：
[[0 1 1 0]]


## 1.2 常见激活函数

激活函数为神经网络引入**非线性**，是深度学习的关键组件。没有激活函数，无论多少层的神经网络都只能表示线性变换。

下面我们逐一学习三种最重要的激活函数：**Sigmoid**、**Tanh** 和 **ReLU**。

### 1.2.1 Sigmoid 激活函数

**公式**：

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

**特点**：
- 输出范围：$(0, 1)$
- 常用于**二分类输出层**（配合交叉熵损失）
- 缺点：两端梯度趋近于 0，容易导致**梯度消失**

**直观理解**：把任意实数"温柔地"压缩到 0~1 之间，非常适合表示概率。

In [2]:
import numpy as np

# 定义 Sigmoid 函数
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 输入数据：2×2 矩阵
t = np.array([[1., 2.], [2., 0.]])

# 计算 Sigmoid
result = sigmoid(t)

print("输入：")
print(t)
print("\nSigmoid 输出：")
print(np.round(result, 6))

# 验证关键性质
print("\n--- Sigmoid 关键性质 ---")
print(f"σ(0)   = {sigmoid(0):.4f}  （中点，输出恰好 0.5）")
print(f"σ(5)   = {sigmoid(5):.4f}  （大正数 → 趋近 1）")
print(f"σ(-5)  = {sigmoid(-5):.4f}  （大负数 → 趋近 0）")

输入：
[[1. 2.]
 [2. 0.]]

Sigmoid 输出：
[[0.731059 0.880797]
 [0.880797 0.5     ]]

--- Sigmoid 关键性质 ---
σ(0)   = 0.5000  （中点，输出恰好 0.5）
σ(5)   = 0.9933  （大正数 → 趋近 1）
σ(-5)  = 0.0067  （大负数 → 趋近 0）



**下面完整地推导一次 sigmoid 函数的导数**  
（从定义开始，一步一步写得很清楚）

Sigmoid 函数定义为：

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

我们求它的导数：$\dfrac{d}{dx} \sigma(x)$

### 方法一：直接用商法则（最常见课堂写法）

把 sigmoid 看成一个分式：

$$
\sigma(x) = \frac{1}{1 + e^{-x}} = \frac{u}{v} \quad \text{其中} \quad
\begin{cases}
u = 1 \\
v = 1 + e^{-x}
\end{cases}
$$

商的导数公式：

$$
\left(\frac{u}{v}\right)' = \frac{u'v - uv'}{v^2}
$$

代入：

- $u' = 0$
- $v' = \dfrac{d}{dx}(1 + e^{-x}) = 0 + e^{-x} \cdot (-1) = -e^{-x}$

所以：

$$
\sigma'(x) = \frac{(0)\cdot(1 + e^{-x}) - (1)\cdot(-e^{-x})}{(1 + e^{-x})^2}
= \frac{0 + e^{-x}}{(1 + e^{-x})^2}
= \frac{e^{-x}}{(1 + e^{-x})^2}
$$

这就是最常见的显式形式之一。

### 方法二：继续化简到 σ(x)(1-σ(x)) 的经典形式（神经网络最常用）

我们现在有：

$$
\sigma'(x) = \frac{e^{-x}}{(1 + e^{-x})^2}
$$

尝试把它用 σ(x) 表示：

注意到：

$$
\sigma(x) = \frac{1}{1 + e^{-x}} \quad \Rightarrow \quad 1 + e^{-x} = \frac{1}{\sigma(x)}
$$

两边平方：

$$
(1 + e^{-x})^2 = \frac{1}{\sigma^2(x)}
$$

所以分母可以换写：

$$
\sigma'(x) = e^{-x} \cdot \frac{1}{(1 + e^{-x})^2} = e^{-x} \cdot \sigma^2(x)
$$

接下来处理分子 e^{-x}：

$$
e^{-x} = (1 + e^{-x}) - 1
$$

所以：

$$
e^{-x} = \frac{1}{\sigma(x)} - 1
$$

代回去：

$$
\sigma'(x) = \left( \frac{1}{\sigma(x)} - 1 \right) \cdot \sigma^2(x)
= \sigma^2(x) \cdot \frac{1}{\sigma(x)} - \sigma^2(x) \cdot 1
= \sigma(x) - \sigma^2(x)
= \sigma(x) (1 - \sigma(x))
$$

**得到最经典、最常用的形式：**

$$
\sigma'(x) = \sigma(x) \big(1 - \sigma(x)\big)
$$

### 小结：两种等价的写法（都正确，场合不同）

| 写法                        | 常用场景                          | 优点                              |
|-----------------------------|------------------------------------|------------------------------------|
| $\sigma(x)(1-\sigma(x))$    | 反向传播代码、梯度计算             | 最简洁、计算最快、无需再次算 exp  |
| $\dfrac{e^{-x}}{(1+e^{-x})^2}$ | 数学推导、手算、理论分析          | 显式，不依赖 σ(x) 的值            |



### 1.2.2 Tanh 激活函数

**公式**：

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = 2\sigma(2x) - 1$$

**特点**：
- 输出范围：$(-1, +1)$
- **零中心**（比 Sigmoid 更有优势，梯度更新不会总是同方向）
- 梯度在 0 附近较大，收敛更快
- 常用于 RNN / LSTM / GRU 的隐藏状态

**直观理解**：比 Sigmoid 更"对称"，正负值都有表达能力，数据分布更友好。

In [ ]:
import numpy as np

# 输入数据
t = np.array([[1., 2.], [2., 0.]])

# 计算 Tanh
result = np.tanh(t)

print("输入：")
print(t)
print("\nTanh 输出：")
print(np.round(result, 6))

# 验证关键性质
print("\n--- Tanh 关键性质 ---")
print(f"tanh(0)   = {np.tanh(0):.4f}   （中点，输出恰好 0）")
print(f"tanh(3)   = {np.tanh(3):.4f}   （大正数 → 趋近 +1）")
print(f"tanh(-3)  = {np.tanh(-3):.4f}  （大负数 → 趋近 -1）")

# 与 Sigmoid 的关系
sigmoid_val = 1 / (1 + np.exp(-2 * 1.0))
print(f"\n验证公式 tanh(x) = 2σ(2x) - 1：")
print(f"  tanh(1)       = {np.tanh(1):.6f}")
print(f"  2σ(2·1) - 1   = {2 * sigmoid_val - 1:.6f}")

### 1.2.3 ReLU 激活函数（目前最主流）

**公式**：

$$\text{ReLU}(x) = \max(0, x)$$

**特点**：
- 输出范围：$[0, +\infty)$
- 计算极快（只是比较和取值）
- 正区域梯度恒为 1，**不存在梯度饱和问题**
- 大幅加速深度网络训练
- 缺点：负值区域梯度为 0，神经元可能"死亡"（Dead ReLU）

**改进变体**：Leaky ReLU、PReLU、ELU、GELU、Swish 等

**一句话总结**：简单高效，几乎所有卷积/全连接隐藏层的默认首选。

In [ ]:
import numpy as np

# 定义 ReLU 函数
def relu(x):
    return np.maximum(0, x)

# 输入数据（包含正数、负数和零）
t = np.array([[1., -2.], [3., 0.]])

# 计算 ReLU
result = relu(t)

print("输入：")
print(t)
print("\nReLU 输出：")
print(result)

# 验证关键性质
print("\n--- ReLU 关键性质 ---")
print(f"ReLU(3)   = {relu(3):.1f}   （正数 → 原样输出）")
print(f"ReLU(0)   = {relu(0):.1f}   （零 → 输出 0）")
print(f"ReLU(-5)  = {relu(-5):.1f}   （负数 → 截断为 0）")

# 对比 Leaky ReLU（允许负值区域有小梯度）
def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

print(f"\n--- Leaky ReLU 对比 ---")
print(f"Leaky ReLU(-5, α=0.01) = {leaky_relu(-5):.2f}  （负数区域保留微小梯度）")

## 1.3 激活函数可视化对比

下面将三种激活函数及其**导数**画在同一张图上，直观比较它们的行为差异。

观察要点：
- **Sigmoid / Tanh**：两端梯度趋近于 0（梯度消失区域）
- **ReLU**：正区域梯度恒为 1，但负区域梯度为 0
- Tanh 比 Sigmoid 更陡峭，说明在 0 附近的梯度更大、收敛更快

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 生成输入数据
x = np.linspace(-5, 5, 300)

# 三种激活函数
y_sigmoid = 1 / (1 + np.exp(-x))
y_tanh = np.tanh(x)
y_relu = np.maximum(0, x)

# 对应的导数
dy_sigmoid = y_sigmoid * (1 - y_sigmoid)
dy_tanh = 1 - y_tanh ** 2
dy_relu = np.where(x > 0, 1.0, 0.0)

# 绘图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：激活函数
axes[0].plot(x, y_sigmoid, label='Sigmoid', linewidth=2)
axes[0].plot(x, y_tanh, label='Tanh', linewidth=2)
axes[0].plot(x, y_relu, label='ReLU', linewidth=2)
axes[0].set_title('激活函数对比', fontsize=14)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-1.5, 5)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)

# 右图：导数
axes[1].plot(x, dy_sigmoid, label="Sigmoid'", linewidth=2)
axes[1].plot(x, dy_tanh, label="Tanh'", linewidth=2)
axes[1].plot(x, dy_relu, label="ReLU'", linewidth=2, linestyle='--')
axes[1].set_title('导数（梯度）对比', fontsize=14)
axes[1].set_xlabel('x')
axes[1].set_ylabel("f'(x)")
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.1, 1.2)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

## 1.4 小结

| 激活函数 | 输出范围 | 零中心 | 梯度消失 | 典型用途 |
|---------|---------|--------|---------|---------|
| Sigmoid | $(0, 1)$ | ✗ | 严重 | 二分类输出层 |
| Tanh | $(-1, 1)$ | ✓ | 较轻 | RNN / LSTM 隐藏层 |
| ReLU | $[0, +\infty)$ | ✗ | 无（正区域） | CNN / FC 隐藏层（默认首选） |

**选择建议**：
- **隐藏层**：优先使用 ReLU（或其变体 Leaky ReLU / GELU）
- **二分类输出层**：使用 Sigmoid
- **多分类输出层**：使用 Softmax（后续章节介绍）
- **RNN 隐藏层**：使用 Tanh